# PhishGuard: 01. Exploratory Data Analysis & Dataset Inspection

## Academic Context & Research Question
> **Research Question:** Can machine learning models identify previously unseen phishing URLs using lexical and structural characteristics of URLs rather than relying only on known malicious blacklists?

In this notebook, we rigorously inspect the benchmark **PhiUSIIL Phishing URL Dataset** (UCI Machine Learning Repository ID: 967), analyze class balance, inspect data types, and categorize feature modalities.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Ensure project root is accessible
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

sns.set_theme(style="whitegrid")
print("Libraries loaded successfully.")

## 1. Load Raw Dataset
We load the uncompressed PhiUSIIL dataset from `data/raw/`.

In [ ]:
raw_csv_path = PROJECT_ROOT / "data" / "raw" / "PhiUSIIL_Phishing_URL_Dataset.csv"
df = pd.read_csv(raw_csv_path)

print(f"Dataset Dimensions: {df.shape[0]:,} rows x {df.shape[1]} columns")
df.head(3)

## 2. Ground Truth Label Distribution
In the raw PhiUSIIL dataset:
- `label == 1`: Legitimate (Benign) URLs
- `label == 0`: Phishing (Malicious) URLs

In cybersecurity and machine learning convention, **1** denotes the target threat class (**Phishing**), while **0** denotes **Benign/Legitimate**.

In [ ]:
raw_counts = df['label'].value_counts()
print("Raw Label Counts:")
print(raw_counts)

# Map to cybersecurity convention: 1 = Phishing, 0 = Legitimate
mapped_target = df['label'].apply(lambda x: 1 if x == 0 else 0)
target_counts = mapped_target.value_counts()

print("\nCybersecurity Standard Target (0 = Legitimate, 1 = Phishing):")
print(f"Legitimate (0): {target_counts[0]:,} ({target_counts[0]/len(df)*100:.2f}%)")
print(f"Phishing   (1): {target_counts[1]:,} ({target_counts[1]/len(df)*100:.2f}%)")

## 3. Class Distribution Visualization

In [ ]:
plt.figure(figsize=(7, 5), dpi=150)
categories = ["Legitimate (Benign)", "Phishing (Malicious)"]
counts = [target_counts[0], target_counts[1]]
colors = ["#2b8a3e", "#c92a2a"]

bars = plt.bar(categories, counts, color=colors, width=0.5, edgecolor="black", linewidth=1.2)
for bar in bars:
    height = bar.get_height()
    pct = (height / sum(counts)) * 100
    plt.text(bar.get_x() + bar.get_width() / 2.0, height + 3000, f"{height:,}\n({pct:.1f}%)", ha="center", va="bottom", fontweight="bold")

plt.title("PhiUSIIL Dataset: Ground Truth Class Distribution", fontsize=13, fontweight="bold", pad=15)
plt.ylabel("Number of Samples")
plt.ylim(0, max(counts) * 1.18)
plt.show()

## 4. URL and Domain Uniqueness
Analyzing URL and Domain uniqueness is critical for designing the **Unseen-Domain Generalization Experiment** in Phase 6.

In [ ]:
total_records = len(df)
unique_urls = df['URL'].nunique()
unique_domains = df['Domain'].nunique()
duplicates = df['URL'].duplicated().sum()

print(f"Total Records:    {total_records:,}")
print(f"Unique URLs:      {unique_urls:,}")
print(f"Duplicate URLs:   {duplicates:,}")
print(f"Unique Domains:   {unique_domains:,}")

## 5. Feature Modality Categorization & Exclusion Rationale

The dataset contains 56 total columns. We classify them into:
1. **Identifiers (3):** `FILENAME`, `URL`, `Domain`
2. **URL Lexical / Structural (23):** Pre-calculated features from the URL string.
3. **Webpage-Dependent Features (29):** HTML, JS, CSS, forms, external references.

### Why Exclude Webpage-Dependent Features?
- **Security Hazard:** Dynamically visiting suspicious URLs exposes clients to zero-day browser exploits and drive-by malware.
- **Zero-Day/Ephemeral Reality:** Phishing campaigns have an average lifespan of hours; dead sites cannot be crawled.
- **Academically Defensible:** The research question focuses exclusively on whether lexical/structural URL features are sufficient for detection.

In [ ]:
webpage_features = [
    "LineOfCode", "LargestLineLength", "HasTitle", "Title",
    "DomainTitleMatchScore", "URLTitleMatchScore", "HasFavicon",
    "Robots", "IsResponsive", "NoOfURLRedirect", "NoOfSelfRedirect",
    "HasDescription", "NoOfPopup", "NoOfiFrame", "HasExternalFormSubmit",
    "HasSocialNet", "HasSubmitButton", "HasHiddenFields",
    "HasPasswordField", "Bank", "Pay", "Crypto", "HasCopyrightInfo",
    "NoOfImage", "NoOfCSS", "NoOfJS", "NoOfSelfRef", "NoOfEmptyRef",
    "NoOfExternalRef"
]

print(f"Identified {len(webpage_features)} webpage-dependent features marked for exclusion.")
for i, f in enumerate(webpage_features, 1):
    print(f"{i:2d}. {f}")